<a href="https://colab.research.google.com/github/Andresflorezdev/DAT-05_ANALISIS_y_Visualizacion_de_Datos-Esumer-Medellin/blob/main/Laboratorio_Clase_9_Estudiantes.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Clase 9 · Agrupaciones y análisis con groupby
**ESUMER / Estud-IA · Johan Steev Muriel Mosquera**

**Objetivo de aprendizaje:** Comprender cómo agrupaciones, agregaciones y denominadores modifican el significado de un indicador y evaluar comparaciones por categorías y periodos.

Institución ficticia. Datos sintéticos. Cada fila de matrículas representa una matrícula, no una persona única. Valor neto calculado después del descuento, no recaudo efectivo.

## 0. Conexión a Google Drive
Antes de ejecutar, crea `Curso_Analitica/Clase_9` en Mi unidad y coloca allí: `Matriculas_enriquecidas.csv`. Abre una copia del notebook en Colab. La siguiente es la primera celda de código.

In [2]:
try:
    from google.colab import drive
except ImportError:
    EN_COLAB = False
    print('Entorno local: se utiliza la carpeta del notebook.')
else:
    EN_COLAB = True
    drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## Cómo avanzar
Ejecuta en orden. Lee la pregunta y la regla antes de programar. Las ayudas son opcionales y los resultados esperados permiten verificar sin copiar una solución. Los controles deben ejecutarse después de completar el ejercicio.

Conserva tu notebook con tu nombre. Si no alcanzas a terminar en la sesión, continúa el mismo recorrido con estas instrucciones. No ejecutes controles de ejercicios que aún no has completado. Al reiniciar, vuelve a ejecutar desde el montaje.


In [6]:
from pathlib import Path
import pandas as pd
import numpy as np
try:
    from IPython.display import display
except ImportError:
    def display(obj):
        print(obj)
CARPETA = Path('/content/drive/MyDrive/Analisis_de_Datos_Esumer/Clase_9') if EN_COLAB else Path.cwd()
SALIDAS = CARPETA / 'Salidas'
SALIDAS.mkdir(exist_ok=True)
print('Pandas:', pd.__version__)
def leer(nombre):
    ruta = CARPETA / nombre
    if not ruta.exists():
        raise FileNotFoundError(f'Revisa la carpeta y el archivo: {ruta}')
    return pd.read_csv(ruta, encoding='utf-8-sig', dtype={'id_matricula':'string', 'id_estudiante':'string'})


Pandas: 2.2.3


## 1. Base preparada y reglas del análisis
La clase 8 dejó 119 matrículas: se separaron dos copias exactas y dos filas con llave contradictoria. No son 119 estudiantes. La base conserva faltantes y banderas.

Para este laboratorio incluimos **todos los estados**. El neto incluye valores conocidos de matrículas activas, finalizadas y canceladas: no es facturación ni recaudo. Si cambias el universo, debes recalcular los denominadores.

Puedes sustituir el archivo por tu salida de la clase 8 si respeta las mismas reglas. Convierte fechas al cargar: un CSV no conserva los tipos de Pandas.

In [7]:
df = leer('Matriculas_enriquecidas.csv')
for col in ['fecha_matricula','fecha_inicio']:
    df[col] = pd.to_datetime(df[col], errors='coerce')
df['mes_matricula'] = df['fecha_matricula'].dt.to_period('M')
assert len(df)==119, 'Revisa las reglas de preparación de la clase 8.'
assert df['id_matricula'].notna().all() and df['id_matricula'].is_unique
display(df.head(3))
display(df[['programa','valor_neto_matricula']].isna().sum())
print('Estudiantes únicos:',df['id_estudiante'].nunique())


,id_matricula,id_estudiante,periodo,programa,sede,edad,fecha_matricula,fecha_inicio,estado,valor_bruto_matricula,...,fecha_matricula_fuente,fecha_inicio_fuente,revisar_fecha,revisar_beca,valor_descuento,valor_neto_matricula,rango_edad,dias_anticipacion,tipo_anticipacion,mes_matricula
0,M0001,E001,2026-1,Analítica,Robledo,58.0,2026-01-08,2026-01-25,Activa,1200000.0,...,2026-01-08,2026-01-25,False,False,240000.0,960000.0,45–59,17.0,Anticipada,2026-01
1,M0002,E002,2026-1,Marketing,Centro,33.0,2026-01-12,2026-01-25,Cancelada,1400000.0,...,2026-01-12,2026-01-25,False,False,0.0,1400000.0,25–34,13.0,Cercana al inicio,2026-01
2,M0003,E003,2026-1,Desarrollo Web,Virtual,NaN,2026-01-07,2026-01-25,Finalizada,1800000.0,...,2026-01-07,2026-01-25,False,False,0.0,1800000.0,Sin información,18.0,Anticipada,2026-01


,0
programa,1
valor_neto_matricula,6


Estudiantes únicos: 60


## Ejercicio 1 · Un resumen con cobertura
Construye `indicadores` agrupando por `periodo` y `programa`. Conserva el programa desconocido con `dropna=False`. Crea: `matriculas` (filas), `estudiantes` (personas únicas), `netos_disponibles` (netos conocidos), `neto_conocido` (suma con al menos un valor conocido) y `neto_promedio`. Ordena por periodo y matrículas descendentes. No rellenes el neto faltante con cero.

**Resultado esperado:** 119 matrículas repartidas entre los grupos; los estudiantes de distintos grupos pueden repetirse. Las cifras exactas se incluyen en `Resultados_esperados.md`.

<details><summary>Ayuda de sintaxis</summary>

```python
tabla.groupby(['categoria_a','categoria_b'], dropna=False, observed=True).agg(
    nombre=('columna','operacion'),
    suma=('valor', lambda s: s.sum(min_count=1))
).reset_index()
```
Usa `size`, `nunique`, `count` y `mean` según la pregunta.

</details>

In [8]:
# COMPLETA AQUÍ
indicadores = df.groupby(['periodo','programa'],dropna=False,observed=True).agg(
    matriculas=('id_matricula','size'), estudiantes=('id_estudiante','nunique'),
    netos_disponibles=('valor_neto_matricula','count'),
    neto_conocido=('valor_neto_matricula',lambda s:s.sum(min_count=1)),
    neto_promedio=('valor_neto_matricula','mean')).reset_index()
indicadores=indicadores.sort_values(['periodo','matriculas'],ascending=[True,False]).reset_index(drop=True)
display(indicadores)

,periodo,programa,matriculas,estudiantes,netos_disponibles,neto_conocido,neto_promedio
0,2026-1,Administración,15,15,13,16900000.0,1.300000e+06
1,2026-1,Marketing,15,15,13,15120000.0,1.163077e+06
2,2026-1,Desarrollo Web,14,14,13,17200000.0,1.323077e+06
3,2026-1,Analítica,13,13,12,13975000.0,1.164583e+06
4,2026-1,Especialización,1,1,1,7200000.0,7.200000e+06
5,2026-1,NaN,1,1,1,1120000.0,1.120000e+06
6,2026-2,Administración,15,15,15,18740000.0,1.249333e+06
7,2026-2,Analítica,15,15,15,19620000.0,1.308000e+06
8,2026-2,Desarrollo Web,15,15,15,19960000.0,1.330667e+06
9,2026-2,Marketing,15,15,15,20160000.0,1.344000e+06


In [9]:
assert indicadores['matriculas'].sum()==len(df), 'Incluye programa sin información.'
esperado=df.groupby(['periodo','programa'],dropna=False,observed=True).agg(matriculas=('id_matricula','size'),estudiantes=('id_estudiante','nunique'),netos_disponibles=('valor_neto_matricula','count'),neto_conocido=('valor_neto_matricula',lambda s:s.sum(min_count=1)),neto_promedio=('valor_neto_matricula','mean')).reset_index()
pd.testing.assert_frame_equal(indicadores.sort_values(['periodo','programa']).reset_index(drop=True),esperado.sort_values(['periodo','programa']).reset_index(drop=True),check_dtype=False)
print('Indicadores y cobertura comprobados.')

Indicadores y cobertura comprobados.


## Ejercicio 2 · Participación de cada sede
Construye `sedes` por periodo y sede con `matriculas` y `estudiantes`. Crea `participacion_pct`: matrículas de la sede / matrículas de ese mismo periodo × 100. Crea `total_periodo` con `transform`. Conserva precisión para validar; redondea solo al mostrar.

**Resultado esperado:** Cada periodo suma 100 %; el total de sus sedes coincide con sus matrículas.

<details><summary>Ayuda de sintaxis</summary>

```python
resumen['total'] = resumen.groupby('periodo')['cantidad'].transform('sum')
```
El denominador no es el total de toda la base.

</details>

In [10]:
# COMPLETA AQUÍ
sedes=df.groupby(['periodo','sede'],dropna=False,observed=True).agg(matriculas=('id_matricula','size'),estudiantes=('id_estudiante','nunique')).reset_index()
sedes['total_periodo']=sedes.groupby('periodo')['matriculas'].transform('sum')
sedes['participacion_pct']=sedes['matriculas']/sedes['total_periodo']*100
display(sedes.round(2))

,periodo,sede,matriculas,estudiantes,total_periodo,participacion_pct
0,2026-1,Centro,20,20,59,33.90
1,2026-1,Robledo,20,20,59,33.90
2,2026-1,Virtual,19,19,59,32.20
3,2026-2,Centro,20,20,60,33.33
4,2026-2,Robledo,20,20,60,33.33
5,2026-2,Virtual,20,20,60,33.33


In [11]:
assert sedes['matriculas'].sum()==119
assert np.allclose(sedes.groupby('periodo')['participacion_pct'].sum(),100)
esperado_sedes=df.groupby(['periodo','sede'],dropna=False).size().rename('matriculas').reset_index()
pd.testing.assert_frame_equal(sedes[esperado_sedes.columns].sort_values(['periodo','sede']).reset_index(drop=True),esperado_sedes.sort_values(['periodo','sede']).reset_index(drop=True),check_dtype=False)
assert np.allclose(sedes['participacion_pct'],sedes['matriculas']/sedes['total_periodo']*100)
print('Participaciones reconciliadas dentro de cada periodo.')

Participaciones reconciliadas dentro de cada periodo.


## Ejercicio 3 · Comparación entre semestres
Construye `comparacion` con una fila por sede y columnas `2026-1`, `2026-2`, `cambio_matriculas`, `variacion_pct` y `cambio_participacion_pp`. La variación usa como base 2026-1. El cambio de participación resta porcentajes y se expresa en puntos porcentuales. Si no existe un volumen base válido y positivo, la variación queda desconocida.

**Resultado esperado:** Tres sedes comparables. No se puede atribuir un cambio a una causa solo con esta tabla.

<details><summary>Ayuda de sintaxis</summary>

```python
resumen.pivot(index='sede', columns='periodo', values='matriculas')
```
Para el denominador: `serie.where(serie.gt(0))`. Otra tabla pivote de participaciones permite calcular puntos porcentuales.

</details>

In [12]:
# COMPLETA AQUÍ
comparacion=sedes.pivot(index='sede',columns='periodo',values='matriculas').copy()
comparacion['cambio_matriculas']=comparacion['2026-2']-comparacion['2026-1']
comparacion['variacion_pct']=comparacion['cambio_matriculas']/comparacion['2026-1'].where(comparacion['2026-1'].gt(0))*100
part=sedes.pivot(index='sede',columns='periodo',values='participacion_pct')
comparacion['cambio_participacion_pp']=part['2026-2']-part['2026-1']
display(comparacion.round(2))

periodo,2026-1,2026-2,cambio_matriculas,variacion_pct,cambio_participacion_pp
sede,,,,,
Centro,20,20,0,0.00,-0.56
Robledo,20,20,0,0.00,-0.56
Virtual,19,20,1,5.26,1.13


In [13]:
assert np.allclose(comparacion['cambio_matriculas'],comparacion['2026-2']-comparacion['2026-1'])
assert np.allclose(comparacion['variacion_pct'],comparacion['cambio_matriculas']/comparacion['2026-1']*100)
p=sedes.pivot(index='sede',columns='periodo',values='participacion_pct')
assert np.allclose(comparacion['cambio_participacion_pp'],p['2026-2']-p['2026-1'])
print('Volumen, variación relativa y puntos porcentuales diferenciados.')

Volumen, variación relativa y puntos porcentuales diferenciados.


## Ejercicio 4 · Auditoría del resumen
Crea `control` con una fila por periodo y columnas `matriculas_base`, `matriculas_resumen`, `netos_base`, `netos_resumen`, `neto_base`, `neto_resumen`. Comprueba cantidades y valores conocidos. Explica por qué no exiges que la suma de estudiantes de los programas sea igual a los estudiantes únicos del periodo.

**Resultado esperado:** Las tres reconciliaciones coinciden. El número de personas únicas no es necesariamente aditivo entre segmentos.

<details><summary>Ayuda de sintaxis</summary>

Construye resúmenes por periodo desde `df` y desde `indicadores`. Puedes alinearlos con `pd.concat(..., axis=1)` porque ambos índices identifican el mismo periodo.

</details>

In [14]:
# COMPLETA AQUÍ
a=df.groupby('periodo').agg(matriculas_base=('id_matricula','size'),netos_base=('valor_neto_matricula','count'),neto_base=('valor_neto_matricula',lambda s:s.sum(min_count=1)))
b=indicadores.groupby('periodo').agg(matriculas_resumen=('matriculas','sum'),netos_resumen=('netos_disponibles','sum'),neto_resumen=('neto_conocido',lambda s:s.sum(min_count=1)))
control=pd.concat([a,b],axis=1)
display(control)

,matriculas_base,netos_base,neto_base,matriculas_resumen,netos_resumen,neto_resumen
periodo,,,,,,
2026-1,59,53,71515000.0,59,53,71515000.0
2026-2,60,60,78480000.0,60,60,78480000.0


In [15]:
assert control['matriculas_base'].eq(control['matriculas_resumen']).all()
assert control['netos_base'].eq(control['netos_resumen']).all()
assert np.allclose(control['neto_base'],control['neto_resumen'])
assert control['matriculas_base'].sum()==len(df)
print('Cantidad, cobertura monetaria y suma comprobadas.')

Cantidad, cobertura monetaria y suma comprobadas.


## 6. Hallazgos y decisión
Escribe tres hallazgos. En cada uno identifica el segmento, el periodo, la cifra y su comparación. Añade una limitación y una información adicional necesaria para decidir.

**Hallazgo 1:** …  
**Hallazgo 2:** …  
**Hallazgo 3:** …  
**Limitación:** …  
**Información adicional:** …

Evita llamar ingresos al neto calculado o afirmar que una categoría causó el cambio. No sumes personas únicas de grupos sin revisar si se superponen.

In [16]:
indicadores.to_csv(SALIDAS/'Indicadores_programa_periodo.csv',index=False,encoding='utf-8-sig')
sedes.to_csv(SALIDAS/'Indicadores_sede_periodo.csv',index=False,encoding='utf-8-sig')
comparacion.to_csv(SALIDAS/'Comparacion_sedes.csv',encoding='utf-8-sig')
control.to_csv(SALIDAS/'Control_indicadores.csv',encoding='utf-8-sig')
print('Salidas guardadas en:',SALIDAS)

Salidas guardadas en: /content/drive/MyDrive/Analisis_de_Datos_Esumer/Clase_9/Salidas


## Si algo falla
| Mensaje | Qué revisar |
|---|---|
| FileNotFoundError | Drive montado, carpeta y nombre exactos. |
| NameError | Ejecución de celdas anteriores y nombre de variable. |
| KeyError | Nombres de columnas con `.columns`. |
| MergeError | Llaves duplicadas y relación esperada. No elimines `validate` para ocultarlo. |
| AssertionError | Lee el mensaje y compara tus resultados con la regla. |
| Totales inflados | Duplicados en la tabla derecha o llaves insuficientes. |

## Referencias oficiales
- [Agrupaciones](https://pandas.pydata.org/docs/user_guide/groupby.html)
- [Integración y concatenación](https://pandas.pydata.org/docs/user_guide/merging.html)
- [Google Drive en Colab](https://colab.research.google.com/notebooks/io.ipynb)
